In [1]:
import numpy as np
from sklearn import preprocessing

print("1. Ładowanie i wstępne przetwarzanie danych...")
# Wczytanie danych z pominięciem nagłówka (jeśli występuje, loadtxt może wyrzucić błąd, ale w instrukcji zakładamy, że go nie ma)
raw_csv_data = np.loadtxt('Audiobooks_data.csv', delimiter=',')

unscaled_inputs_all = raw_csv_data[:, 1:-1]
targets_all = raw_csv_data[:, -1]

# Przetasowanie danych przed balansowaniem
shuffled_indices = np.arange(unscaled_inputs_all.shape[0])
np.random.shuffle(shuffled_indices)

unscaled_inputs_all = unscaled_inputs_all[shuffled_indices]
targets_all = targets_all[shuffled_indices]

print("2. Balansowanie zbioru danych...")
num_one_targets = int(np.sum(targets_all))
zero_targets_counter = 0
indices_to_remove = []

for i in range(targets_all.shape[0]):
    if targets_all[i] == 0:
        zero_targets_counter += 1
        if zero_targets_counter > num_one_targets:
            indices_to_remove.append(i)

unscaled_inputs_equal_priors = np.delete(unscaled_inputs_all, indices_to_remove, axis=0)
targets_equal_priors = np.delete(targets_all, indices_to_remove, axis=0)

print("3. Normalizacja i tasowanie...")
scaled_inputs = preprocessing.scale(unscaled_inputs_equal_priors)

shuffled_indices = np.arange(scaled_inputs.shape[0])
np.random.shuffle(shuffled_indices)

shuffled_inputs = scaled_inputs[shuffled_indices]
shuffled_targets = targets_equal_priors[shuffled_indices]

print("4. Podział na zestawy: Trening, Walidacja, Test...")
samples_count = shuffled_inputs.shape[0]

train_samples_count = int(0.8 * samples_count)
validation_samples_count = int(0.1 * samples_count)
test_samples_count = samples_count - train_samples_count - validation_samples_count

train_inputs = shuffled_inputs[:train_samples_count]
train_targets = shuffled_targets[:train_samples_count]

validation_inputs = shuffled_inputs[train_samples_count:train_samples_count+validation_samples_count]
validation_targets = shuffled_targets[train_samples_count:train_samples_count+validation_samples_count]

test_inputs = shuffled_inputs[train_samples_count+validation_samples_count:]
test_targets = shuffled_targets[train_samples_count+validation_samples_count:]

print(f"Trening (jedynki / suma / %): {np.sum(train_targets)} / {train_samples_count} / {np.sum(train_targets) / train_samples_count:.2f}")
print(f"Walidacja (jedynki / suma / %): {np.sum(validation_targets)} / {validation_samples_count} / {np.sum(validation_targets) / validation_samples_count:.2f}")
print(f"Test (jedynki / suma / %): {np.sum(test_targets)} / {test_samples_count} / {np.sum(test_targets) / test_samples_count:.2f}")

print("5. Zapisywanie do plików .npz...")
np.savez('Audiobooks_data_train', inputs=train_inputs, targets=train_targets)
np.savez('Audiobooks_data_validation', inputs=validation_inputs, targets=validation_targets)
np.savez('Audiobooks_data_test', inputs=test_inputs, targets=test_targets)
print("Gotowe! Pliki zostały wygenerowane.")

1. Ładowanie i wstępne przetwarzanie danych...
2. Balansowanie zbioru danych...
3. Normalizacja i tasowanie...
4. Podział na zestawy: Trening, Walidacja, Test...
Trening (jedynki / suma / %): 1787.0 / 3579 / 0.50
Walidacja (jedynki / suma / %): 233.0 / 447 / 0.52
Test (jedynki / suma / %): 217.0 / 448 / 0.48
5. Zapisywanie do plików .npz...
Gotowe! Pliki zostały wygenerowane.


In [2]:
import numpy as np
import tensorflow as tf

print("1. Ładowanie przetworzonych danych...")
npz = np.load('Audiobooks_data_train.npz')
train_inputs = npz['inputs'].astype(np.float32)
train_targets = npz['targets'].astype(np.int32)

npz = np.load('Audiobooks_data_validation.npz')
validation_inputs, validation_targets = npz['inputs'].astype(np.float32), npz['targets'].astype(np.int32)

npz = np.load('Audiobooks_data_test.npz')
test_inputs, test_targets = npz['inputs'].astype(np.float32), npz['targets'].astype(np.int32)

print("2. Budowa zoptymalizowanego modelu (>90% accuracy)...")
input_size = 10
output_size = 2
# ZMIANA: Zwiększono liczbę neuronów do 128 dla lepszej skuteczności
hidden_layer_size = 128

model = tf.keras.Sequential([
    tf.keras.layers.Dense(hidden_layer_size, activation='relu'),
    tf.keras.layers.Dense(hidden_layer_size, activation='relu'),
    # ZMIANA: Dodano trzecią warstwę ukrytą
    tf.keras.layers.Dense(hidden_layer_size, activation='relu'),
    tf.keras.layers.Dense(output_size, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# ZMIANA: Mniejszy batch_size dla lepszej generalizacji
batch_size = 64
max_epochs = 100
# ZMIANA: Wydłużona tolerancja wczesnego zatrzymywania
early_stopping = tf.keras.callbacks.EarlyStopping(patience=5)

print("3. Start treningu...")
model.fit(train_inputs,
          train_targets,
          batch_size=batch_size,
          epochs=max_epochs,
          callbacks=[early_stopping],
          validation_data=(validation_inputs, validation_targets),
          verbose=1)

print("\n4. Ewaluacja modelu na zestawie testowym:")
test_loss, test_accuracy = model.evaluate(test_inputs, test_targets)
print('\nTest loss: {0:.2f}. Test accuracy: {1:.2f}%'.format(test_loss, test_accuracy * 100.))

1. Ładowanie przetworzonych danych...
2. Budowa zoptymalizowanego modelu (>90% accuracy)...
3. Start treningu...
Epoch 1/100
56/56 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.7460 - loss: 0.5000 - val_accuracy: 0.7651 - val_loss: 0.4106
Epoch 2/100
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7910 - loss: 0.3996 - val_accuracy: 0.7919 - val_loss: 0.3964
Epoch 3/100
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7893 - loss: 0.3958 - val_accuracy: 0.7942 - val_loss: 0.3829
Epoch 4/100
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7963 - loss: 0.3823 - val_accuracy: 0.7875 - val_loss: 0.3846
Epoch 5/100
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7980 - loss: 0.3817 - val_accuracy: 0.8009 - val_loss: 0.3720
Epoch 6/100
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8069 - loss: 0.3694 - val_accuracy: 0.7718 - val_loss: 0.3787
Epoch 7/100
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8078 - loss: 0.3706 - val_accuracy: 0.8076 - val_loss: 0.360